In [90]:
#!pip install torchvision torch

# LMUL Accuracy Testing on MNIST Dataset via MLP task

Given a python-level LMUL implementation (so not hardware), how accurate is LMUL in an MLP in relation to normal multiplication?

Note: This is purely on the inference level (for now), training is still with exact math

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

#load MNIST things
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST('.', train=True, download=True, transform=transforms.ToTensor()),
    batch_size=128, shuffle=True
)
test_loader = torch.utils.data.DataLoader(
    datasets.MNIST('.', train=False, transform=transforms.ToTensor()),
    batch_size=1000
)

#LMUL, based on the paper definition (below)

def lmul(a, b, M=7):
    """
    L-Mul approximation based on the paper formula:
    L-Mul(x, y) = (s1 ⊕ s2) × 2^(e1 + e2 - b) × (1 + m1 + m2 + 2^(-L(M)))
    
    where L(M) = { M,     if M ≤ 3
                 { 3,     if M = 4  
                 { 4,     if M > 4
    
    a, b: Input tensors
    M: Precision parameter (bits), assumption is arbitrarily 7
    """
    a, b = a.float(), b.float()
    
    #the XOR (⊕) sign thing in front
    s1 = torch.sign(a)
    s2 = torch.sign(b)
    s = s1 * s2
    
    #mantissa and exponent
    #frexp returns: x = m × 2^e where m ∈ [0.5, 1)
    m1, e1 = torch.frexp(torch.abs(a))
    m2, e2 = torch.frexp(torch.abs(b))
    #So m_frexp = (1 + m_standard) / 2, thus m_standard = 2*m_frexp - 1
    m1 = 2 * m1 - 1  # Convert [0.5, 1) to [0, 1)
    m2 = 2 * m2 - 1  # Convert [0.5, 1) to [0, 1)
    
    #L(M) 
    if M <= 3:
        L = M
    elif M == 4:
        L = 3
    else:  # M > 4
        L = 4
    
    #bias b (for normalized mantissa, b = 1) (i think?)
    b = 1
    #L-Mul formula: result = s × 2^(e1 + e2 - b) × (1 + m1 + m2 + 2^(-L))
    exponent = e1 + e2 - b
    mantissa = 1 + m1 + m2 + 2**(-L)
    
    #if mantissa >= 2, normalize
    carry = (mantissa >= 2).float()
    mantissa = torch.where(carry == 1, mantissa / 2, mantissa)
    exponent = exponent + carry
    #result
    exponent = exponent.to(torch.int32)
    out = s * torch.ldexp(mantissa, exponent)
    
    return torch.nan_to_num(out, nan=0.0, posinf=float('inf'), neginf=float('-inf'))


def lmul_linear(x, W, b=None, M=7):
    #actual use of lmul for forward pass
    prod = lmul(x.unsqueeze(1), W.unsqueeze(0), M)  # [B,O,I]
    out = prod.sum(dim=2)
    if b is not None:
        out += b
    return out

#MLP for PURELY ACCURACY METRICS
class MLP(nn.Module):
    def __init__(self, use_lmul=False, M=7):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 10)
        self.use_lmul = use_lmul
        self.M = M

    def forward(self, x):
        x = x.view(-1, 28*28)
        if self.use_lmul:
            x = F.relu(lmul_linear(x, self.fc1.weight, self.fc1.bias, self.M))
            x = lmul_linear(x, self.fc2.weight, self.fc2.bias, self.M)
        else:
            x = F.relu(self.fc1(x))
            x = self.fc2(x)
        return F.log_softmax(x, dim=1)

#stereotypical training setup for MLP
def train_model(model, opt, loader, epochs=2):
    model.train()
    for _ in range(epochs):
        for data, target in loader:
            opt.zero_grad()
            loss = F.nll_loss(model(data), target)
            loss.backward()
            opt.step()

#acc test
def test_acc(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in loader:
            pred = model(data).argmax(dim=1)
            correct += (pred == target).sum().item()
            total += len(target)
    return 100 * correct / total


#baseline trainin 
model = MLP(use_lmul=False)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
train_model(model, opt, train_loader, epochs=10)
torch.save(model.state_dict(), "mlp_baseline.pth")
#normal acc eval (no lmul)
print(f"Baseline accuracy: {test_acc(model, test_loader):.2f}%")
#lmul
#model_lmul = MLP(use_lmul=True, M=7)
#model_lmul.load_state_dict(model.state_dict())
#print(f"L-MUL accuracy: {test_acc(model_lmul, test_loader):.2f}%")

In [94]:
#im paranoid about the actual results but I dont think this provides a lot of info (we care more about relative correctness between mults than objective corre
a = torch.randn(10000000)
b = torch.randn(10000000)
prod = a * b
approx = lmul(a, b)
print("mean(|LMul|)/mean(|mul|) =", approx.abs().mean().item() / prod.abs().mean().item())

mean(|LMul|)/mean(|mul|) = 2.5482086650511984


In [105]:
a = torch.tensor([1])
b = torch.tensor([1])
lmul(a, b)

tensor([4.1250])

In [115]:
a = torch.tensor([5])
b = torch.tensor([8])
lmul(a, b)

tensor([84.])

In [30]:
import torch

ckpt = torch.load("mlp_baseline.pth", map_location="cpu")

# adjust key if needed
# common names: 'fc.bias', 'classifier.bias', 'linear.bias'
for k in ckpt.keys():
    if "bias" in k:
        print(k, ckpt[k].shape)

# Example: final linear layer
b2 = ckpt["fc2.bias"]          # shape: [10]
b2 = b.float()
print("Bias fc2:", b2)

fc1.bias torch.Size([128])
fc2.bias torch.Size([10])
Bias fc2: tensor([-0.1671, -0.0483,  0.0003, -0.0492,  0.0615,  0.1586, -0.0793,  0.0042,
         0.0033, -0.0216])


/tmp/ipykernel_408/4083981604.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load("mlp_baseline.pth", map_location="cpu")


In [31]:
import torch
import torch.nn.functional as F

def load_verilog_mem_autoscale(filepath, num_classes=10, verbose=True):
    """
    Load a Verilog .mem file containing 16-bit fixed-point logits and automatically scale.
    
    Returns:
        logits: float32 tensor (N, num_classes)
        log_probs: log-softmax probabilities
        preds: predicted classes
    """
    # --- read hex ---
    with open(filepath) as f:
        hex_vals = [line.strip() for line in f if line.strip()]
    
    # --- parse as int32 first ---
    u32 = torch.tensor([int(h, 16) for h in hex_vals], dtype=torch.int32)
    
    # --- convert to signed int16 ---
    logits_raw = torch.where(u32 >= 0x8000, u32 - 0x10000, u32).float()
    
    # --- automatically choose scale ---
    max_abs = logits_raw.abs().max()
    
    # heuristics: Q4.12 if max < 20, else Q8.8
    if max_abs > 20:
        scale = 4096.0  # Q4.12
        if verbose:
            print(f"Detected Q4.12 (scale={scale}), max raw value={max_abs.item():.1f}")
    else:
        scale = 256.0   # Q8.8
        if verbose:
            print(f"Detected Q8.8 (scale={scale}), max raw value={max_abs.item():.1f}")
    
    logits = logits_raw / scale
    
    # --- reshape ---
    assert logits.numel() % num_classes == 0
    logits = logits.view(-1, num_classes)
    
    # --- log_softmax + predictions ---
    if b2 is not None:
        logits = logits + b2.unsqueeze(0)
    log_probs = F.log_softmax(logits, dim=1)
    preds = log_probs.argmax(dim=1)
    
    return logits, log_probs, preds

# -------------------------------
# Usage
# -------------------------------
logits, log_probs, preds = load_verilog_mem_autoscale("data/results.mem", num_classes=10)

print("First 10 logits:\n", logits[:1])
print("First 10 log_probs:\n", log_probs[:1])
print("Prediction counts:\n", torch.bincount(preds))


Detected Q4.12 (scale=4096.0), max raw value=19328.0
First 10 logits:
 tensor([[ 3.4594, -4.3623, -4.3217,  3.5553, -4.2940, -4.1632, -4.3608,  3.6295,
         -4.3085,  3.6752]])
First 10 log_probs:
 tensor([[-1.5106, -9.3323, -9.2917, -1.4147, -9.2640, -9.1332, -9.3308, -1.3405,
         -9.2785, -1.2948]])
Prediction counts:
 tensor([  0,   0,   0,  29,  71, 793,   0,   5,  23,  79])


In [32]:
preds

tensor([9, 3, 5, 5, 5, 5, 5, 5, 5, 4, 5, 5, 9, 5, 5, 5, 5, 9, 3, 5, 5, 5, 5, 5,
        5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5, 4, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 3,
        5, 5, 5, 5, 5, 5, 3, 5, 5, 5, 9, 5, 9, 5, 5, 5, 4, 5, 5, 5, 5, 5, 9, 5,
        3, 5, 5, 9, 5, 5, 5, 9, 9, 5, 5, 9, 5, 5, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5,
        5, 5, 5, 5, 5, 5, 5, 4, 5, 9, 4, 5, 5, 5, 5, 5, 5, 9, 5, 5, 5, 5, 5, 8,
        5, 5, 5, 5, 5, 9, 3, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 7, 5, 5,
        4, 5, 5, 4, 5, 4, 5, 8, 5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 5, 5, 5, 5, 5, 8,
        5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
        4, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
        5, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 4, 9, 4, 5, 5, 5, 5, 9, 4, 5, 5, 5,
        5, 5, 5, 9, 5, 5, 5, 5, 5, 3, 5, 5, 9, 5, 9, 5, 4, 5, 4, 3, 5, 5, 4, 9,
        8, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 5, 5, 5, 5, 5, 4, 3, 8,
        5, 3, 5, 4, 5, 4, 5, 8, 5, 5, 5,

In [33]:
for i in range(10):
    max_logit_idx = (logits.argmax(dim=1) == i).sum()
    print(f"Class {i}: {max_logit_idx.item()} times predicted")

Class 0: 0 times predicted
Class 1: 0 times predicted
Class 2: 0 times predicted
Class 3: 29 times predicted
Class 4: 71 times predicted
Class 5: 793 times predicted
Class 6: 0 times predicted
Class 7: 5 times predicted
Class 8: 23 times predicted
Class 9: 79 times predicted


In [11]:
h = "bad9"
u = torch.tensor([int(h, 16)], dtype=torch.uint16)
print(u.view(torch.bfloat16).float())

tensor([-0.0017])
